# Sparkle V2 — Fase 5+6: Target de desatención + entrenamiento del LSTM baseline

Este notebook asume que ya corriste la Fase 4 (archivos `train_consolidado.npz`, `validation_consolidado.npz`, `test_consolidado.npz` en Drive).

**Decisión de target (ya cerrada con tu compañero):** usamos **Boredom** colapsado a 3 niveles, porque:
- Engagement está desbalanceado ~95% en 2 de sus 4 clases (inservible tal cual).
- Boredom tiene correlación moderada con Engagement (-0.42), no es la misma señal invertida — aporta información real.
- Colapsando Boredom {0} / {1} / {2,3} queda razonablemente balanceado (~45% / 30% / 24% en Train).

**Qué hace este notebook:**
1. Carga los tensores consolidados de la Fase 4.
2. Construye el target de 3 niveles a partir de Boredom.
3. Normaliza los features (fit del scaler solo en Train, coherente con buenas prácticas — nunca fitear con datos de val/test).
4. Calcula class weights para compensar el desbalance restante durante el entrenamiento (en vez de sumar otro dataset, que decidimos dejar como posible extensión futura).
5. Entrena un LSTM baseline (arquitectura simple, 2 capas).
6. Evalúa en Test con accuracy, F1 por clase y matriz de confusión.
7. Guarda el modelo entrenado y el scaler en Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
DATASET_DIR = f'{PROJECT_DIR}/dataset_final'
MODELS_DIR = f'{PROJECT_DIR}/checkpoints'
os.makedirs(MODELS_DIR, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 1. Cargar los datasets consolidados de la Fase 4

In [ ]:
def load_split(split_name):
    data = np.load(f'{DATASET_DIR}/{split_name}_consolidado.npz', allow_pickle=True)
    return {
        'X': data['X'],
        'clip_ids': data['clip_ids'],
        'engagement': data['engagement'],
        'boredom': data['boredom'],
        'confusion': data['confusion'],
        'frustration': data['frustration'],
    }

train = load_split('train')
val = load_split('validation')
test = load_split('test')

print('Train:', train['X'].shape)
print('Validation:', val['X'].shape)
print('Test:', test['X'].shape)

## 2. Construir el target de 3 niveles a partir de Boredom

- Nivel 0 (Atento): Boredom original = 0
- Nivel 1 (Desatención leve): Boredom original = 1
- Nivel 2 (Desatención marcada): Boredom original = 2 o 3

In [ ]:
def collapse_boredom(boredom_array):
    return np.where(boredom_array == 0, 0, np.where(boredom_array == 1, 1, 2)).astype(np.int32)

y_train = collapse_boredom(train['boredom'])
y_val = collapse_boredom(val['boredom'])
y_test = collapse_boredom(test['boredom'])

CLASS_NAMES = ['Atento', 'Desatencion_leve', 'Desatencion_marcada']

for name, y in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    counts = pd.Series(y).value_counts().sort_index()
    pct = (counts / counts.sum() * 100).round(1)
    print(f'{name}:')
    for cls in range(3):
        c = counts.get(cls, 0)
        p = pct.get(cls, 0.0)
        print(f'  {CLASS_NAMES[cls]}: {c} ({p}%)')
    print()

## 3. Normalización de features

EAR está en una escala ~0.1-0.4 y yaw/pitch en grados (~-90 a 90) — sin normalizar, el LSTM le va a dar de entrada mucho más peso relativo a los ángulos que a la apertura ocular, no porque sean más importantes sino por la escala. El `StandardScaler` se ajusta **solo con Train** (nunca con val/test, para no filtrar información).

In [ ]:
n_frames, n_features = train['X'].shape[1], train['X'].shape[2]

scaler = StandardScaler()
X_train_flat = train['X'].reshape(-1, n_features)
scaler.fit(X_train_flat)

def scale_X(X):
    shape = X.shape
    X_flat = X.reshape(-1, n_features)
    X_scaled = scaler.transform(X_flat)
    return X_scaled.reshape(shape)

X_train = scale_X(train['X'])
X_val = scale_X(val['X'])
X_test = scale_X(test['X'])

print('Media por feature (Train, post-escalado, debe ser ~0):', X_train.reshape(-1, n_features).mean(axis=0).round(3))
print('Std por feature (Train, post-escalado, debe ser ~1):', X_train.reshape(-1, n_features).std(axis=0).round(3))

## 4. Class weights

En vez de sumar otro dataset para compensar el desbalance (que decidimos dejar como posible extensión futura), probamos primero con class weighting — penaliza más los errores en las clases minoritarias durante el entrenamiento.

In [ ]:
class_weights_array = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}
print('Class weights:', class_weight_dict)

## 5. Arquitectura del modelo

LSTM simple de 2 capas — baseline antes de comparar contra arquitecturas más sofisticadas (ViBED-Net, etc. que citás en el estado del arte).

In [ ]:
def build_model(n_frames, n_features, n_classes=3):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_frames, n_features)),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_model(n_frames, n_features)
model.summary()

## 6. Entrenamiento

Con `EarlyStopping` (corta si no mejora en 10 épocas) y `ModelCheckpoint` (guarda el mejor modelo directo en Drive, por si se corta la sesión).

In [ ]:
checkpoint_path = f'{MODELS_DIR}/lstm_boredom_v1_best.keras'

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_loss', save_best_only=True
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

## 7. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/lstm_boredom_v1_training_curves.png', dpi=100)
plt.show()

## 8. Evaluación en Test

**Ojo con la comparación:** los benchmarks que citás en la tesis (64-67% LSTM, 73% ViBED-Net) son para clasificación de **Engagement en 4 niveles**, no para nuestro target de **Boredom en 3 niveles** — no son directamente comparables. Los reportamos igual como referencia de orden de magnitud, pero en la tesis hay que aclarar esta diferencia de tarea para no comparar peras con manzanas.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_accuracy:.4f}')
print(f'Test loss: {test_loss:.4f}')

y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/lstm_boredom_v1_confusion_matrix.png', dpi=100)
plt.show()

## 9. Guardar modelo final + scaler

El scaler es tan importante como el modelo: sin él, cualquier feature nuevo (por ejemplo, en producción dentro de la extensión) va a estar en la escala equivocada.

In [ ]:
import joblib

model.save(f'{MODELS_DIR}/lstm_boredom_v1_final.keras')
joblib.dump(scaler, f'{MODELS_DIR}/scaler_v1.pkl')

print('Modelo guardado en:', f'{MODELS_DIR}/lstm_boredom_v1_final.keras')
print('Scaler guardado en:', f'{MODELS_DIR}/scaler_v1.pkl')

## Checkpoint de la Fase 5+6

Pegame acá:
- El `classification_report` completo de la sección 8 (accuracy, precision/recall/F1 por clase).
- La matriz de confusión (o una descripción de qué se confunde con qué).
- Si el entrenamiento cortó por early stopping o llegó a las 100 épocas.

Con eso decidimos si el baseline es suficiente para la tesis o si conviene: (a) ajustar la arquitectura, (b) probar features adicionales (roll, velocidad de cambio de yaw/pitch entre frames), o (c) recién ahí evaluar sumar otro dataset.

In [ ]:

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
DATASET_DIR = f'{PROJECT_DIR}/dataset_final'
MODELS_DIR = f'{PROJECT_DIR}/checkpoints'
os.makedirs(MODELS_DIR, exist_ok=True)

CLASS_NAMES = ['Atento', 'Desatencion_leve', 'Desatencion_marcada']

In [ ]:
def load_split(split_name):
    data = np.load(f'{DATASET_DIR}/{split_name}_consolidado.npz', allow_pickle=True)
    return {
        'X': data['X'],
        'boredom': data['boredom'],
    }

train = load_split('train')
val = load_split('validation')
test = load_split('test')

def collapse_boredom(boredom_array):
    return np.where(boredom_array == 0, 0, np.where(boredom_array == 1, 1, 2)).astype(np.int32)

y_train = collapse_boredom(train['boredom'])
y_val = collapse_boredom(val['boredom'])
y_test = collapse_boredom(test['boredom'])

print('Shapes:', train['X'].shape, val['X'].shape, test['X'].shape)

In [ ]:
majority_class = pd.Series(y_train).value_counts().idxmax()
trivial_preds = np.full_like(y_test, majority_class)
trivial_acc = accuracy_score(y_test, trivial_preds)
print(f'Clase mayoritaria (Train): {CLASS_NAMES[majority_class]}')
print(f'Accuracy del baseline trivial en Test: {trivial_acc:.4f}')
print('\nCualquier modelo que propongamos tiene que superar claramente este número para ser útil.')

In [ ]:
def summarize_clip(X):
    # X: (n_clips, n_frames, n_features) -> (n_clips, n_features*4)
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    minimum = X.min(axis=1)
    maximum = X.max(axis=1)
    return np.concatenate([mean, std, minimum, maximum], axis=1)

X_train_summary = summarize_clip(train['X'])
X_val_summary = summarize_clip(val['X'])
X_test_summary = summarize_clip(test['X'])

scaler_summary = StandardScaler()
X_train_summary_scaled = scaler_summary.fit_transform(X_train_summary)
X_val_summary_scaled = scaler_summary.transform(X_val_summary)
X_test_summary_scaled = scaler_summary.transform(X_test_summary)

print('Shape de features resumen:', X_train_summary_scaled.shape)

In [ ]:
print('--- Regresión Logística ---')
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, multi_class='multinomial')
logreg.fit(X_train_summary_scaled, y_train)
y_pred_logreg = logreg.predict(X_test_summary_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred_logreg):.4f}')
print(classification_report(y_test, y_pred_logreg, target_names=CLASS_NAMES, digits=3))

In [ ]:
print('--- Random Forest ---')
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_summary_scaled, y_train)
y_pred_rf = rf.predict(X_test_summary_scaled)
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Importancia de features del Random Forest — ayuda a ver qué variables cargan señal real
feature_labels = []
for stat in ['mean', 'std', 'min', 'max']:
    for f in ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch']:
        feature_labels.append(f'{f}_{stat}')

importances = pd.Series(rf.feature_importances_, index=feature_labels).sort_values(ascending=False)
print('Top 10 features más importantes según Random Forest:')
print(importances.head(10))